In [1]:
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List, Optional
from pypdf import PdfReader
from dotenv import load_dotenv

# Load OPENAI_API_KEY from a local .env file (see .env.example) - never hard-code a key
load_dotenv()

client = OpenAI()

class PaperData(BaseModel):
    title: str = Field(description="The title of the paper")
    abstract_summary: List[str] = Field(description="One sentence summary of the abstract")
    intro_backgrounds_summary: str = Field(description="One sentence summary of the intro and backgrounds sections")
    methods_summary: str = Field(description="One sentence summary of the methods section")
    results_summary: str = Field(description="One sentence summary of the results section")
    discussion_summary: str = Field(description="One sentence summary of the discussion section")
    

def load_pdf_text(file_path):
    '''Loads text from a PDF file.'''
    # creating a pdf reader object
    reader = PdfReader(file_path)

    # extracting text from page
    text = "\n\n".join([page.extract_text() for page in reader.pages])
    
    return text    

In [2]:
import os, urllib.request

def download_pdf(url, dest):
    """Download a PDF to `dest` if it is not already there (cross-platform; no wget)."""
    if os.path.exists(dest):
        print(f"Already present: {dest}")
        return
    os.makedirs(os.path.dirname(dest) or ".", exist_ok=True)
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as r, open(dest, "wb") as f:
        f.write(r.read())
    print(f"Downloaded: {dest}")

download_pdf("https://arxiv.org/pdf/2510.26493", "../assets/paper.pdf")

Already present: ../assets/paper.pdf


In [3]:
paper_raw_text = load_pdf_text("../assets/paper.pdf")

response = client.responses.parse(
    model="gpt-5.6-luna",
    input=f"extract the information from the paper: {paper_raw_text}",
    text_format=PaperData
)

print(response)

ParsedResponse(id='resp_054f79410b570b11006ab0fc66a3bc87d289a5dbfd2700c36b', created_at=1789983846.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.6-luna', object='response', output=[ResponseReasoningItem(id='rs_054f79410b570b11006ab0fc67880c87d29c84b580c8972a9c', summary=[], type='reasoning', content=[], encrypted_content='gAAAAABqsPxqHHynEPXjuColmZx79i1Nwtu_IkgBM3RB03j87Og99qCYaEgobfLJM67asIwg0SAWIBSPNGtLKnwtNchUf42YFpsz4_v8xs9isb0PV9qA_f4KLmBP2mOFBWeaWM5YEyIo2FqiL56qeMfw4xvbOpFqOHZ-UHeZcqxEvTtAn53xR2udDDmM5NlrwPxI3ijlo-2f7OA13QVH08A1K86RczRbk-mvaA3V0zhGqcXwpQQZEK0y0QVqFhkM7Y1vh-cHCA9eCrpezgqW8N8y7NrTJjk_DxZLlcpEFSS-zzZHXaMNOBjr5xGq_CieXQZdMagq_75V5-ASj8r9qwuhm5E4POSIHkFQMqJS_UqnKOj3h5y7RC5SXzvDbY7NRPSxDtEC9VvT7M9A4wggTk7nNCd0pLsBEGQ9gbnvkGWovjifhWwYX58AxW-TrQOVzB0mjrKwQ_BWkKooemWaqYBZRppMPlkNM4Ojzg3bEpwSyEJRVXwB2WI-Ab98kT0nHEGduOb9cDHefHrDn5GDZO5YKCErkeaJfwdF5NwhYe8sS9lG0Nm6DGI6S0RpVp4krXJXEdeq3uKUQ-2kyhSM_2gwxrIEtVLSU_FSEbrbUez-ijpdMuTrxpaDBTOEb

In [4]:
response.output_parsed

PaperData(title='Context Engineering 2.0: The Context of Context Engineering', abstract_summary=['The paper presents context engineering as the systematic design, collection, storage, management, and use of contextual information to reduce the gap between human intentions and machine understanding, tracing its evolution from early human–computer interaction to future human-level and superhuman AI systems.'], intro_backgrounds_summary='The introduction situates context engineering within the longer history of ubiquitous computing, context-aware systems, HCI, prompting, retrieval, tool use, and agent memory, arguing that increasing machine intelligence progressively reduces the human effort required to translate high-entropy intentions into machine-readable context.', methods_summary='The authors develop a conceptual and theoretical framework that formally defines entities, interactions, context, and context engineering, compares four evolutionary stages, reviews historical practices, an

In [5]:
from IPython.display import Markdown, display

md = f"""
# {response.output_parsed.title}

## Abstract Summary
{'; '.join(response.output_parsed.abstract_summary)}

## Introduction & Backgrounds Summary
{response.output_parsed.intro_backgrounds_summary}

## Methods Summary
{response.output_parsed.methods_summary}

## Results Summary
{response.output_parsed.results_summary}

## Discussion Summary
{response.output_parsed.discussion_summary}
"""

display(Markdown(md))


# Context Engineering 2.0: The Context of Context Engineering

## Abstract Summary
The paper presents context engineering as the systematic design, collection, storage, management, and use of contextual information to reduce the gap between human intentions and machine understanding, tracing its evolution from early human–computer interaction to future human-level and superhuman AI systems.

## Introduction & Backgrounds Summary
The introduction situates context engineering within the longer history of ubiquitous computing, context-aware systems, HCI, prompting, retrieval, tool use, and agent memory, arguing that increasing machine intelligence progressively reduces the human effort required to translate high-entropy intentions into machine-readable context.

## Methods Summary
The authors develop a conceptual and theoretical framework that formally defines entities, interactions, context, and context engineering, compares four evolutionary stages, reviews historical practices, and organizes design considerations around context collection and storage, management, and usage.

## Results Summary
The paper characterizes Context Engineering 1.0 as structured, sensor-driven, human-prepared input and Context Engineering 2.0 as agent-centered processing of natural-language and multimodal context, while outlining future 3.0 and 4.0 stages involving human-level understanding and proactive or superhuman context construction.

## Discussion Summary
The discussion identifies challenges involving multimodal collection, lifelong storage, context overload, long-context efficiency, relevance selection, system stability, evaluation, privacy, and proactive need inference, and proposes a future semantic operating system that actively organizes, updates, explains, and manages context over time.


In [6]:
import pandas as pd
import os

def upsert_paper_to_csv(paper_data, csv_path="papers_database.csv", key_column="title"):
    """
    Inserts or updates a paper's data in a CSV file database. If the file does not exist,
    it creates it. If the file exists, it updates the entry if the key exists, or appends 
    the new paper if not.

    Parameters:
        paper_data: an object with attributes corresponding to extracted fields
        csv_path: output file path for CSV
        key_column: column name to use as unique identifier (default: "title")
    """
    # Prepare data dictionary, flattening abstract_summary if it's a list
    paper_dict = {
        "title": getattr(paper_data, "title", ""),
        "abstract_summary": "; ".join(paper_data.abstract_summary) if isinstance(getattr(paper_data, "abstract_summary", ""), list) else getattr(paper_data, "abstract_summary", ""),
        "intro_backgrounds_summary": getattr(paper_data, "intro_backgrounds_summary", ""),
        "methods_summary": getattr(paper_data, "methods_summary", ""),
        "results_summary": getattr(paper_data, "results_summary", ""),
        "discussion_summary": getattr(paper_data, "discussion_summary", ""),
    }

    # If CSV exists, load and update if key matches; otherwise create new
    if os.path.isfile(csv_path):
        df = pd.read_csv(csv_path)
        # Check if key (e.g., title) already in db
        if paper_dict[key_column] in df[key_column].values:
            # Update existing entry
            df.loc[df[key_column] == paper_dict[key_column], list(paper_dict.keys())] = list(paper_dict.values())
            updated = True
        else:
            # Append if not found
            df = pd.concat([df, pd.DataFrame([paper_dict])], ignore_index=True)
            updated = False
    else:
        # Create new dataframe
        df = pd.DataFrame([paper_dict])
        updated = False

    df.to_csv(csv_path, index=False)
    if updated:
        print(f"Updated entry in '{csv_path}' for {key_column}='{paper_dict[key_column]}'")
    else:
        print(f"Added new entry to '{csv_path}' for {key_column}='{paper_dict[key_column]}'")

# Example usage:
upsert_paper_to_csv(response.output_parsed)

Added new entry to 'papers_database.csv' for title='Context Engineering 2.0: The Context of Context Engineering'


In [7]:
import pandas as pd

df = pd.read_csv("papers_database.csv")

df

,title,abstract_summary,intro_backgrounds_summary,methods_summary,results_summary,discussion_summary
0,Context Engineering 2.0: The Context of Contex...,The paper presents context engineering as the ...,The introduction situates context engineering ...,The authors develop a conceptual and theoretic...,The paper characterizes Context Engineering 1....,The discussion identifies challenges involving...


In [8]:
download_pdf("https://arxiv.org/pdf/1301.3781", "../assets/paper3.pdf")

Already present: ../assets/paper3.pdf


In [9]:
def extract_paper_data_workflow(paper_path: str):
    paper_raw_text = load_pdf_text(paper_path)

    response = client.responses.parse(
        model="gpt-5.6-luna",
        input=f"extract the information from the paper: {paper_raw_text}",
        text_format=PaperData
    )
    
    output_parsed = response.output_parsed

    print(output_parsed)
    
    upsert_paper_to_csv(output_parsed)
    
    return output_parsed

extract_paper_data_workflow("../assets/paper3.pdf")

title='Efficient Estimation of Word Representations in Vector Space' abstract_summary=['The paper introduces the CBOW and Skip-gram architectures for efficiently learning high-quality word vectors from massive text corpora, demonstrating improved syntactic and semantic relationship performance at substantially lower computational cost.'] intro_backgrounds_summary='The paper motivates distributed word representations as a scalable alternative to atomic word indices and reviews neural language-model approaches, emphasizing the need for methods capable of training high-dimensional vectors on billions of words while preserving linear semantic and syntactic relationships.' methods_summary='The authors compare feedforward and recurrent neural language models with two simpler log-linear architectures—Continuous Bag-of-Words, which predicts a target word from surrounding context, and Continuous Skip-gram, which predicts surrounding words from a target word—using hierarchical softmax, stochasti

PaperData(title='Efficient Estimation of Word Representations in Vector Space', abstract_summary=['The paper introduces the CBOW and Skip-gram architectures for efficiently learning high-quality word vectors from massive text corpora, demonstrating improved syntactic and semantic relationship performance at substantially lower computational cost.'], intro_backgrounds_summary='The paper motivates distributed word representations as a scalable alternative to atomic word indices and reviews neural language-model approaches, emphasizing the need for methods capable of training high-dimensional vectors on billions of words while preserving linear semantic and syntactic relationships.', methods_summary='The authors compare feedforward and recurrent neural language models with two simpler log-linear architectures—Continuous Bag-of-Words, which predicts a target word from surrounding context, and Continuous Skip-gram, which predicts surrounding words from a target word—using hierarchical softm

In [10]:
import pandas as pd


df = pd.read_csv("papers_database.csv")
df

,title,abstract_summary,intro_backgrounds_summary,methods_summary,results_summary,discussion_summary
0,Context Engineering 2.0: The Context of Contex...,The paper presents context engineering as the ...,The introduction situates context engineering ...,The authors develop a conceptual and theoretic...,The paper characterizes Context Engineering 1....,The discussion identifies challenges involving...
1,Efficient Estimation of Word Representations i...,The paper introduces the CBOW and Skip-gram ar...,The paper motivates distributed word represent...,The authors compare feedforward and recurrent ...,"On semantic-syntactic analogy benchmarks, Skip...",The authors conclude that simple CBOW and Skip...


In [11]:
tool_schema = {
    "type": "function",
    "name": "extract_paper_data_workflow",
    "description": "Extracts data from a paper",
    "parameters": {
        "type": "object",
        "properties": {
            "paper_path": {
                "type": "string",
                "description": "The path to the paper"
            }
        },
        "required": ["paper_path"],
    }
}

tool_schema

{'type': 'function',
 'name': 'extract_paper_data_workflow',
 'description': 'Extracts data from a paper',
 'parameters': {'type': 'object',
  'properties': {'paper_path': {'type': 'string',
    'description': 'The path to the paper'}},
  'required': ['paper_path']}}

In [12]:
download_pdf("https://arxiv.org/pdf/2412.14161", "./test_agent_paper.pdf")

Already present: ./test_agent_paper.pdf


In [13]:
response = client.responses.create(
    model="gpt-5.6-luna",
    instructions="You are an extraction agent, users will give you paths to files for pdfs\
                  and you will extract the data from the paper using the:\
                  extract_paper_data function.",
    tools=[tool_schema],
    input="Extract data from this paper: ./test_agent_paper.pdf"
)

response.output_text

''

In [14]:
response.output

[ResponseReasoningItem(id='rs_0ea6da665f023cd5006ab0fc70f39087d2a539c2c3ccca620b', summary=[], type='reasoning', content=[], encrypted_content='gAAAAABqsPxxqyt6JQ8VuWZLde5pbY2YqUEuiKrGmRVnvf0qZD37PGtQet2mabPadLya2LsfROYmEW4YPU9J0tTe09-9gA0iw1sfDFgVcVAnShzK5enyn0roGHyXpRhUrZGdME83t6vizC5UnP8AVjFI2dgaI-nIyW2lhAGyLl9sEtBkoxeqKwuqnQJ-teXgd07lg-zDnEHOyfmIsXRQveGYaDCoG8e6Higk6zsk6N46IcFJsSozrG5enEChC_GFRGty_gevcioR_O3CA0Rd1yNY5MdiTujGzybxgbDehkkBwDCuw9c5wXOnHRTgyqnPaf2Vaf_PbaNxFjSAzMGOIaGtRwQtfTjNdyjZGmRjRH5Tt5CRILZfik5kKcrpEPDkbeH-eQdh_6eQowDmj_ci7Ke59NkIcUNsjjReZXjtIYE8Kcpp5KyeLMs7mEkipPpgJNjahq35ZdCqDwNJXhL0YYGeghN1BWGvdHhdvfRLDMMdNVd2IOc1n1ZNjdDwJ-rH0X3-3CNE0Bm7A--szPq9lQtA9y1j8pJAlwbyXNmgBTdqOJ_HMya3eEP-OdVLqPMa4NlxgWvV04pAraIxDZ1OpMvLm9wx5NuDpgtLXJADSUJgiF7MBF2KgndAuiM4YLRoYr6ozivhUCKtTmq5vs8oldAAtXsc0rp4DOgPmvzLBw6bik3MarEOuomso1kSeGYZOvYSng2anABKRZqQ8reW5Abq-NYO3gXXLVuT3cVWI6fb2t5DZZosUjO5bWSHMvdYZeE2c0BGTg6HkxQlcFGCMU4X4IXKS5mo_skaCKEkdwhAeA3ZdM2aIPWWM57yXCwIadIPejkuWreFEAJi8x1uN2K6